# BỐI CẢNH PHÂN TÍCH - GIAI ĐOẠN MỞ RỘNG 


## I. TỪ BÀI TOÁN QUẢN TRỊ ĐẾN CÂU HỎI PHÂN TÍCH 

Khi lượng đơn hàng bắt đầu tăng lên và dịch vụ giao hàng đã đi vào ổn định, Danny đứng trước một bài toán mới: **làm sao bán được nhiều hơn mà không để chi phí nguyên liệu “ngốn” hết lợi nhuận vừa kiếm được?** Ở giai đoạn khởi nghiệp, mỗi đồng tiết kiệm được từ nguyên liệu có thể là ranh giới giữa lời và lỗ. Và ngược lại, nếu không hiểu khách hàng thích thêm gì, ghét bỏ gì, Danny sẽ bỏ lỡ cơ hội tăng doanh thu từ những “tùy chỉnh nhỏ” mà khách sẵn sàng trả tiền.

Câu chuyện không chỉ dừng ở việc mua bao nhiêu, bán bao nhiêu. Nó đi sâu vào **kỹ thuật thực đơn (Menu Engineering)** và **quản trị chuỗi cung ứng (Supply Chain)**. Danny cần biết chính xác: Một chiếc Meatlovers tiêu chuẩn ngốn bao nhiêu gam thịt xông khói? Khi khách yêu cầu “thêm phô mai, bỏ hành”, liệu đầu bếp có biết ngay công thức thực tế của chiếc bánh đó là gì không? Và quan trọng hơn, liệu bộ phận kế toán có dễ dàng tính được chi phí nguyên liệu cho từng đơn hàng để điều chỉnh giá bán một cách linh hoạt?

Tư duy của giai đoạn này là **“cắt giảm mỡ thừa” và “bơm thêm cơ bắp”**. Cắt giảm những topping thường xuyên bị khách từ chối để tránh lãng phí. Bơm vào những topping được ưa chuộng bằng cách thiết kế các gói thêm (extras) có tính phí. Đồng thời, xây dựng một hệ thống hiển thị công thức thông minh cho bếp và kế toán, giúp giảm sai sót và tăng tốc độ xử lý đơn.


| Câu hỏi quản trị | Các câu hỏi phân tích |
|------------------|-----------------------|
| Topping nào nên được upsell để tăng doanh thu? | C.1 – Topping được thêm nhiều nhất |
| Topping nào gây lãng phí nhất? | C.2 – Topping bị loại bỏ nhiều nhất |
| Làm thế nào để bếp không bị nhầm đơn khi khách tùy chỉnh? | C.3 – Công thức chuẩn từng pizza <br> C.4 – Danh sách nguyên liệu thực tế sau tùy chỉnh (hiển thị cho bếp) |
| Giá vốn thực tế của mỗi đơn hàng là bao nhiêu? | C.5 – Tổng lượng từng nguyên liệu đã dùng trong các đơn thành công |

 

## II. PHÂN TÍCH VÀ ĐỀ XUẤT HÀNH ĐỘNG 

### NHÓM 1: CƠ HỘI TĂNG DOANH THU TỪ TOPPING (UPSELL)

In [0]:
%sql
USE Pizza_Runner;

##### C.1. What was the most commonly added extra?

(Topping được thêm nhiều nhất)

In [0]:
%sql
WITH extras_of_pizza_in_each_order AS (
	SELECT 
			c.order_id,
			c.pizza_id, 
			ex.topping_id AS extras
	FROM destination.customer_orders c
	INNER JOIN destination.customer_orders_extras ex ON c.customer_orders_id = ex.customer_orders_id 
)
, used_extras AS (
	SELECT
		    e.extras ,
			COUNT(e.pizza_id) AS numbers
	FROM extras_of_pizza_in_each_order e
	GROUP BY e.extras 
	-- TRONG CTE, không cho ORDER BY ngoại trừ việc dùng kèm với TOP, OFFSET ,...
)
, rank_extras AS (
	SELECT 
		u.extras,
		u.numbers,
		DENSE_RANK() OVER(ORDER BY u.numbers DESC) AS rank_used_numbers_of_extras
	FROM used_extras u
)
SELECT *
FROM rank_extras
WHERE rank_used_numbers_of_extras = 1
-- NGHĨA LÀ BACON là topping được thêm nhiều nhất 

-- Nhưng cách làm này rườm rà và mù ý nghĩa report do extras chỉ là số id trong hệ thống chứ không phải tên BACON


extras,numbers,rank_used_numbers_of_extras
1,4,1


In [0]:
%sql
WITH topping_counts AS (
	SELECT 
			t.topping_name AS most_common_extra, 
	        COUNT(t.topping_id) AS total_added_times
	FROM destination.customer_orders_extras ex
	INNER JOIN destination.pizza_toppings t ON ex.topping_id = t.topping_id 
	GROUP BY t.topping_name
)
SELECT 
	most_common_extra,
	total_added_times
FROM topping_counts
QUALIFY RANK() OVER (ORDER BY total_added_times DESC) = 1

most_common_extra,total_added_times
Bacon,4


### NHÓM 2: CẮT GIẢM LÃNG PHÍ NGUYÊN 

##### C.2. What was the most common exclusion?

(Topping bị loại bỏ nhiều nhất)

In [0]:
%sql
WITH topping_counts AS (
	SELECT 
			t.topping_name AS most_common_exclusion, 
	        COUNT(t.topping_id) AS total_excluded_times
	FROM destination.customer_orders_exclusions exc
	INNER JOIN destination.pizza_toppings t ON exc.topping_id = t.topping_id 
	GROUP BY t.topping_name
)
SELECT 
	most_common_exclusion,
	total_excluded_times
FROM topping_counts
QUALIFY RANK() OVER (ORDER BY total_excluded_times DESC) = 1

most_common_exclusion,total_excluded_times
Cheese,4


### NHÓM 3: CHUẨN HÓA HIỆN THỊ ĐƠN HÀNG CHO NHÀ BẾP  

##### C.3. What are the standard ingredients for each pizza?

(Công thức chuẩn từng pizza)

In [0]:
%sql
-- Vì dữ liệu đã được chuẩn hóa trước khi đưa vào destination nên chỉ cần viết câu lệnh select chứ không cần xử lý chuỗi nữa

SELECT 
		n.pizza_name,
		t.topping_name
FROM destination.pizza_recipes p
INNER JOIN destination.pizza_names n ON p.pizza_id = n.pizza_id
INNER JOIN destination.pizza_toppings t ON p.toppings = t.topping_id
ORDER BY n.pizza_name ASC,
		 t.topping_name ASC

-- KQ:
-- pizza_id = 1 có toppings: 1, 2, 3, 4, 5, 6, 8, 10 
-- pizza_id = 2 có toppings: 4, 6, 7, 9, 11, 12 
  
-- Nghĩa là: 
-- Meatlovers có topping là : Bacon, BBQ Sauce, Beef, Cheese, Chicken, Mushrooms, Pepperoni, Salami
-- Vegetarian có topping là : Cheese, Mushrooms, Onions, Peppers, Tomatoes, Tomato Sauce
  


pizza_name,topping_name
Meatlovers,BBQ Sauce
Meatlovers,Bacon
Meatlovers,Beef
Meatlovers,Cheese
Meatlovers,Chicken
Meatlovers,Mushrooms
Meatlovers,Pepperoni
Meatlovers,Salami
Vegetarian,Cheese
Vegetarian,Mushrooms


##### C.4. 

**Generate an order item for each record in the customers_orders table in the format of one of the following:**

* **"Meat Lovers"**

* **"Meat Lovers - Exclude Beef"**

* **"Meat Lovers - Extra Bacon"**

* **"Meat Lovers - Exclude Cheese, Bacon - Extra Mushroom, Peppers"**

In [0]:
%sql
-- Để giải quyết bài toán này, tôi sẽ chia làm 3 chặng: 
-- Chặng 01: Xử lý topping bỏ đi (exclusions)
-- Chặng 02: Xử lý topping thêm vào (extras)
-- Chặng 03: Tổng hợp lại
---- Nếu ở bước xử lý dữ liệu, chuẩn hóa vào destination ta đã STRING_SPLIT để xử lý vấn đề đa trị, nhưng khi làm báo cáo 
---- cần tổng hợp lại, thì dùng STRING_AGG 
----  LEFT JOIN , lấy orders ở customer_orders rồi LEFT JOIN vs exclusions, extras. Lý do là vì để giữ lại những pizza khách đặt mà không thựuc hiện tùy chỉnh 

-- Chặng 01: Xử lý topping bỏ đi (exclusions)
WITH ExclusionsCTE AS (
		SELECT
				exc.customer_orders_id ,
				CONCAT('Exclude ', ARRAY_JOIN(COLLECT_LIST(p.topping_name), ', ')) AS exclusiion_text 
		FROM  destination.customer_orders_exclusions exc
		INNER JOIN destination.pizza_toppings p ON exc.topping_id = p.topping_id 
		-- Mức độ chi tiết của bảng là mỗi hàng là một cái bánh pizza 
		GROUP BY exc.customer_orders_id 
), 
-- Chặng 02: Xử lý topping thêm vào (extras)
ExtrasCTE AS (
		SELECT 
				ext.customer_orders_id,
				CONCAT('Extra ', ARRAY_JOIN(COLLECT_LIST(p.topping_name), ', ')) AS extra_text
		FROM destination.customer_orders_extras ext 
		INNER JOIN destination.pizza_toppings p ON ext.topping_id = p.topping_id  
		GROUP BY ext.customer_orders_id

)
-- Chặng 03: Tổng hợp lại
SELECT
		co.customer_orders_id,
		co.order_id,
		CONCAT(
			pn.pizza_name,
			COALESCE(CONCAT(' - ', exc.exclusiion_text), ''),
			COALESCE(CONCAT(' - ', ext.extra_text), '')
		) AS order_item
FROM destination.customer_orders co
JOIN destination.pizza_names pn ON co.pizza_id = pn.pizza_id
LEFT JOIN ExclusionsCTE exc	ON co.customer_orders_id = exc.customer_orders_id
LEFT JOIN ExtrasCTE ext ON co.customer_orders_id = ext.customer_orders_id 


customer_orders_id,order_id,order_item
1,1,Meatlovers
2,2,Meatlovers
3,3,Meatlovers
4,3,Vegetarian
5,4,Meatlovers - Exclude Cheese
6,4,Meatlovers - Exclude Cheese
7,4,Vegetarian - Exclude Cheese
8,5,Meatlovers - Extra Bacon
9,6,Vegetarian
10,7,Vegetarian - Extra Bacon


##### C.5. 
**Generate an alphabetically ordered comma separated ingredient list for each pizza order from the customer_orders table and add a 2x in front of any relevant ingredients** 

**(e.g. "Meat Lovers: 2xBacon, Beef, ... , Salami")**


In [0]:
%sql

;WITH RECURSIVE total_available_toppings AS(
	SELECT 
			 co.order_id , 
			 co.customer_orders_id , 
			 co.pizza_id ,
			 tp.topping_id AS total_topping_id,
			 tp.topping_name
	FROM destination.customer_orders co
	CROSS JOIN destination.pizza_toppings tp
), specific_toppings_for_pizza AS (
	SELECT 
			tap.order_id,
			tap.customer_orders_id,
			tap.pizza_id,
			tap.total_topping_id,
			tap.topping_name,
			rp.toppings AS topping_id_for_specific_pizza,
			CASE
				WHEN rp.toppings IS NOT NULL THEN 1
				ELSE 0
			END AS base_topping,
			extras.topping_id AS extras_topping_id_for_specific_pizza,
			CASE
				WHEN extras.topping_id IS NOT NULL THEN 1
				ELSE 0
			END AS count_extras_tp,
			exclusions.topping_id AS exclusions_topping_id_for_specific_pizza,
			CASE
				WHEN exclusions.topping_id IS NOT NULL THEN 1
				ELSE 0
			END AS count_exclusion_tp
	FROM total_available_toppings tap 
	LEFT JOIN destination.pizza_recipes rp ON ( tap.pizza_id = rp.pizza_id )
										   AND (tap.total_topping_id = rp.toppings) 
	LEFT JOIN destination.customer_orders_extras extras ON  ( tap.customer_orders_id = extras.customer_orders_id )
														AND	(tap.total_topping_id = extras.topping_id)
	LEFT JOIN destination.customer_orders_exclusions exclusions ON  ( tap.customer_orders_id = exclusions.customer_orders_id )
														AND	(tap.total_topping_id = exclusions.topping_id)
)
, final_CTE AS (
	SELECT 
			s.order_id,
			s.customer_orders_id,
			s.pizza_id,
			s.total_topping_id,
			s.topping_id_for_specific_pizza,
			s.base_topping,
			s.extras_topping_id_for_specific_pizza,
			s.count_extras_tp,
			s.exclusions_topping_id_for_specific_pizza,
			s.count_exclusion_tp,
			(s.base_topping + s.count_extras_tp - s.count_exclusion_tp) AS final_count,
			s.topping_name
	FROM specific_toppings_for_pizza s
)
, formatted_toppings AS (
	SELECT 
			f.order_id,
			f.customer_orders_id,
			-- Nếu hệ số là 1, thì chỉ ghi topping_name, không cần ghi hệ số trước
			-- Còn nêú > 1 , ví dụ như 2, 3 thì chuyển 2, 3 thành VARCHAR rồi nối với 'x' 
			-- sql xuống WHERE rồi lên SELECT là sau cùng, nên trường hợp final_count = 0 đến đây không còn nữa 
			CASE
				WHEN f.final_count > 1 THEN CONCAT(CAST(f.final_count AS STRING), 'x', f.topping_name)
				ELSE f.topping_name
			END AS topping_text,
			f.topping_name
	FROM final_CTE f
	INNER JOIN destination.pizza_names pn ON f.pizza_id = pn.pizza_id 
	-- Nếu hệ số là 0, không ghi vào công thức của mỗi chiếc bánh  (có thể được tinh chỉnh bởi khách)
	WHERE f.final_count > 0
	-- Để cho thứ tự xuất hiện công thức của từng chiếc bánh đồng nhất 
	ORDER BY f.order_id ASC, f.customer_orders_id ASC, f.topping_name ASC
) 
SELECT
			ft.order_id,
			ft.customer_orders_id,
			-- Nguyên liệu làm từng chiếc bánh trong mỗi đơn hàng 
			ARRAY_JOIN(
				-- ARRAY_SORT đảm bảo thứ tự ABC cố định, không lung tung
				ARRAY_SORT(COLLECT_LIST(ft.topping_text)),
				-- những hàng thành 1 hàng được nối với nhau bằng dấu cộng thể hiện cách ghi công thức cho từng cái bánh được đặt đẹp 
				'+ '
			) AS specific_recipe_for_each_pizza
FROM formatted_toppings ft
GROUP BY 
		ft.order_id, 
		ft.customer_orders_id  
ORDER BY ft.order_id ASC, 
		 ft.customer_orders_id ASC
 


order_id,customer_orders_id,specific_recipe_for_each_pizza
1,1,BBQ Sauce+ Bacon+ Beef+ Cheese+ Chicken+ Mushrooms+ Pepperoni+ Salami
2,2,BBQ Sauce+ Bacon+ Beef+ Cheese+ Chicken+ Mushrooms+ Pepperoni+ Salami
3,3,BBQ Sauce+ Bacon+ Beef+ Cheese+ Chicken+ Mushrooms+ Pepperoni+ Salami
3,4,Cheese+ Mushrooms+ Onions+ Peppers+ Tomato Sauce+ Tomatoes
4,5,BBQ Sauce+ Bacon+ Beef+ Chicken+ Mushrooms+ Pepperoni+ Salami
4,6,BBQ Sauce+ Bacon+ Beef+ Chicken+ Mushrooms+ Pepperoni+ Salami
4,7,Mushrooms+ Onions+ Peppers+ Tomato Sauce+ Tomatoes
5,8,2xBacon+ BBQ Sauce+ Beef+ Cheese+ Chicken+ Mushrooms+ Pepperoni+ Salami
6,9,Cheese+ Mushrooms+ Onions+ Peppers+ Tomato Sauce+ Tomatoes
7,10,Bacon+ Cheese+ Mushrooms+ Onions+ Peppers+ Tomato Sauce+ Tomatoes


### NHÓM 4: TÍNH GIÁ VỐN HÀNG BÁN THỰC TẾ

##### C.6. What is the total quantity of each ingredient used in all delivered pizzas sorted by most frequent first?

(Tổng lượng từng nguyên liệu đã dùng trong các đơn thành công)

In [0]:
%sql
;WITH RECURSIVE total_available_toppings AS(
	SELECT 
			 co.order_id , 
			 co.customer_orders_id , 
			 co.pizza_id ,
			 tp.topping_id AS total_topping_id,
			 tp.topping_name
	FROM destination.customer_orders co
	CROSS JOIN destination.pizza_toppings tp
), specific_toppings_for_pizza AS (
	SELECT 
			tap.order_id,
			tap.customer_orders_id,
			tap.pizza_id,
			tap.total_topping_id,
			tap.topping_name,
			rp.toppings AS topping_id_for_specific_pizza,
			CASE
				WHEN rp.toppings IS NOT NULL THEN 1
				ELSE 0
			END AS base_topping,
			extras.topping_id AS extras_topping_id_for_specific_pizza,
			CASE
				WHEN extras.topping_id IS NOT NULL THEN 1
				ELSE 0
			END AS count_extras_tp,
			exclusions.topping_id AS exclusions_topping_id_for_specific_pizza,
			CASE
				WHEN exclusions.topping_id IS NOT NULL THEN 1
				ELSE 0
			END AS count_exclusion_tp
	FROM total_available_toppings tap 
	LEFT JOIN destination.pizza_recipes rp ON ( tap.pizza_id = rp.pizza_id )
										   AND (tap.total_topping_id = rp.toppings) 
	LEFT JOIN destination.customer_orders_extras extras ON  ( tap.customer_orders_id = extras.customer_orders_id )
														AND	(tap.total_topping_id = extras.topping_id)
	LEFT JOIN destination.customer_orders_exclusions exclusions ON  ( tap.customer_orders_id = exclusions.customer_orders_id )
														AND	(tap.total_topping_id = exclusions.topping_id)
)
, final_CTE AS (
	SELECT 
			s.order_id,
			s.customer_orders_id,
			s.pizza_id,
			s.total_topping_id,
			s.topping_id_for_specific_pizza,
			s.base_topping,
			s.extras_topping_id_for_specific_pizza,
			s.count_extras_tp,
			s.exclusions_topping_id_for_specific_pizza,
			s.count_exclusion_tp,
			(s.base_topping + s.count_extras_tp - s.count_exclusion_tp) AS final_count,
			s.topping_name
	FROM specific_toppings_for_pizza s
) 
SELECT  
		f.topping_name,
		SUM(f.final_count) AS used_quantiies
FROM	final_CTE f
INNER JOIN destination.runner_orders ro ON f.order_id = ro.order_id
WHERE ro.cancellation IS NULL 
GROUP BY f.topping_name
ORDER BY used_quantiies DESC

topping_name,used_quantiies
Bacon,12
Mushrooms,11
Cheese,10
Salami,9
Chicken,9
Pepperoni,9
Beef,9
BBQ Sauce,8
Tomato Sauce,3
Peppers,3


### TỐI ƯU C.5, C.6 BẰNG CÁCH TẠO VIEW 

In [0]:
%sql

-- ĐỂ TRÁNH VIỆC CÂU 5, 6 PHẢI PASTE LẠI MỘT PHẦN CTE KHÁ DÀI, TẠO VIEW CHUNG RỒI TRUY VẤN TỪ VIEW ĐÓ 
CREATE VIEW destination.vw_pizza_ingredient_matrix AS 
WITH total_available_toppings AS(
	SELECT 
			 co.order_id , 
			 co.customer_orders_id , 
			 co.pizza_id ,
			 tp.topping_id AS total_topping_id,
			 tp.topping_name
	FROM destination.customer_orders co
	CROSS JOIN destination.pizza_toppings tp
), specific_toppings_for_pizza AS (
	SELECT 
			tap.order_id,
			tap.customer_orders_id,
			tap.pizza_id,
			tap.total_topping_id,
			tap.topping_name,
			rp.toppings AS topping_id_for_specific_pizza,
			CASE
				WHEN rp.toppings IS NOT NULL THEN 1
				ELSE 0
			END AS base_topping,
			extras.topping_id AS extras_topping_id_for_specific_pizza,
			CASE
				WHEN extras.topping_id IS NOT NULL THEN 1
				ELSE 0
			END AS count_extras_tp,
			exclusions.topping_id AS exclusions_topping_id_for_specific_pizza,
			CASE
				WHEN exclusions.topping_id IS NOT NULL THEN 1
				ELSE 0
			END AS count_exclusion_tp
	FROM total_available_toppings tap 
	LEFT JOIN destination.pizza_recipes rp ON ( tap.pizza_id = rp.pizza_id )
										   AND (tap.total_topping_id = rp.toppings) 
	LEFT JOIN destination.customer_orders_extras extras ON  ( tap.customer_orders_id = extras.customer_orders_id )
														AND	(tap.total_topping_id = extras.topping_id)
	LEFT JOIN destination.customer_orders_exclusions exclusions ON  ( tap.customer_orders_id = exclusions.customer_orders_id )
														AND	(tap.total_topping_id = exclusions.topping_id)
) 
SELECT 
		s.order_id,
		s.customer_orders_id,
		s.pizza_id,
		s.total_topping_id,
		s.topping_id_for_specific_pizza,
		s.base_topping,
		s.extras_topping_id_for_specific_pizza,
		s.count_extras_tp,
		s.exclusions_topping_id_for_specific_pizza,
		s.count_exclusion_tp,
		(s.base_topping + s.count_extras_tp - s.count_exclusion_tp) AS final_count,
		s.topping_name
FROM specific_toppings_for_pizza s

In [0]:
%sql
SELECT *
FROM destination.vw_pizza_ingredient_matrix

order_id,customer_orders_id,pizza_id,total_topping_id,topping_id_for_specific_pizza,base_topping,extras_topping_id_for_specific_pizza,count_extras_tp,exclusions_topping_id_for_specific_pizza,count_exclusion_tp,final_count,topping_name
1,1,1,1,1,1,null,0,null,0,1,Bacon
2,2,1,2,2,1,null,0,null,0,1,BBQ Sauce
3,3,1,3,3,1,null,0,null,0,1,Beef
3,4,2,4,4,1,null,0,null,0,1,Cheese
4,5,1,5,5,1,null,0,null,0,1,Chicken
4,6,1,6,6,1,null,0,null,0,1,Mushrooms
4,7,2,7,7,1,null,0,null,0,1,Onions
5,8,1,8,8,1,null,0,null,0,1,Pepperoni
6,9,2,9,9,1,null,0,null,0,1,Peppers
7,10,2,10,null,0,null,0,null,0,0,Salami


In [0]:
%sql
-- C.5
SELECT
			f.order_id,
			f.customer_orders_id,
			-- Nguyên liệu làm từng chiếc bánh trong mỗi đơn hàng 
			ARRAY_JOIN(
				-- ARRAY_SORT đảm bảo thứ tự ABC cố định
				ARRAY_SORT(
					COLLECT_LIST(
						-- Nếu ở tiền xử lý dùng STRING_SPLIT để giải quyết đa trị
						-- Còn để trình bày dashboard đệp, thì dùng ARRAY_JOIN(), AGG trong aggregate, là từ nhiều hàng thành 1 
						CASE
							-- Nếu hệ số là 1, thì chỉ ghi topping_name, không cần ghi hệ số trước
							-- Còn nêú > 1 , ví dụ như 2, 3 thì chuyển 2, 3 thành STRING rồi nối với 'x' 
							-- sql xuống WHERE rồi lên SELECT là sau cùng, nên trường hợp final_count = 0 đến đây không còn nữa 
							WHEN f.final_count > 1 THEN CONCAT(CAST(f.final_count AS STRING), 'x', f.topping_name)
							ELSE f.topping_name
						END
					)
				),
				-- những hàng thành 1 hàng được nối với nhau bằng dấu cộng thể hiện cách ghi công thức cho từng cái bánh được đặt đẹp 
				'+ '
			) 
			-- Để cho thứ tự xuất hiện công thức của từng chiếc bánh đồng nhất 
			AS specific_recipe_for_each_pizza
FROM destination.vw_pizza_ingredient_matrix f
INNER JOIN destination.pizza_names pn ON f.pizza_id = pn.pizza_id 
-- Nếu hệ số là 0, không ghi vào công thức của mỗi chiếc bánh  (có thể được tinh chỉnh bởi khách)
WHERE f.final_count > 0
GROUP BY 
		f.order_id, 
		f.customer_orders_id  
ORDER BY f.order_id ASC, 
		 f.customer_orders_id ASC


order_id,customer_orders_id,specific_recipe_for_each_pizza
1,1,BBQ Sauce+ Bacon+ Beef+ Cheese+ Chicken+ Mushrooms+ Pepperoni+ Salami
2,2,BBQ Sauce+ Bacon+ Beef+ Cheese+ Chicken+ Mushrooms+ Pepperoni+ Salami
3,3,BBQ Sauce+ Bacon+ Beef+ Cheese+ Chicken+ Mushrooms+ Pepperoni+ Salami
3,4,Cheese+ Mushrooms+ Onions+ Peppers+ Tomato Sauce+ Tomatoes
4,5,BBQ Sauce+ Bacon+ Beef+ Chicken+ Mushrooms+ Pepperoni+ Salami
4,6,BBQ Sauce+ Bacon+ Beef+ Chicken+ Mushrooms+ Pepperoni+ Salami
4,7,Mushrooms+ Onions+ Peppers+ Tomato Sauce+ Tomatoes
5,8,2xBacon+ BBQ Sauce+ Beef+ Cheese+ Chicken+ Mushrooms+ Pepperoni+ Salami
6,9,Cheese+ Mushrooms+ Onions+ Peppers+ Tomato Sauce+ Tomatoes
7,10,Bacon+ Cheese+ Mushrooms+ Onions+ Peppers+ Tomato Sauce+ Tomatoes


In [0]:
%sql
-- C.6
SELECT  
		f.topping_name,
		SUM(f.final_count) AS used_quantiies
FROM	destination.vw_pizza_ingredient_matrix f
INNER JOIN destination.runner_orders ro ON f.order_id = ro.order_id
WHERE ro.cancellation IS NULL 
GROUP BY f.topping_name
ORDER BY used_quantiies DESC

topping_name,used_quantiies
Bacon,12
Mushrooms,11
Cheese,10
Salami,9
Chicken,9
Pepperoni,9
Beef,9
BBQ Sauce,8
Tomato Sauce,3
Peppers,3


## III. TỔNG KẾT 

## PHỤ LỤC: KHÁC BIỆT CÚ PHÁP GIỮA T‑SQL VÀ DATABRICKS SQL

| Vấn đề | T‑SQL (SQL Server) | Databricks SQL (Spark SQL) | Lý do |
|--------|-------------------|----------------------------|-------|
| **Lấy dòng đồng hạng cao nhất** | `SELECT TOP 1 WITH TIES … ORDER BY cnt DESC` | `QUALIFY RANK() OVER (ORDER BY cnt DESC) = 1` | Databricks không hỗ trợ `TOP WITH TIES`; `QUALIFY` lọc trực tiếp trên hàm cửa sổ. |
| **Gom chuỗi từ nhiều dòng** | `STRING_AGG(expr, ', ') WITHIN GROUP (ORDER BY col)` | `ARRAY_JOIN(ARRAY_SORT(COLLECT_LIST(expr)), ', ')` | `STRING_AGG` và `WITHIN GROUP` không có trong Spark SQL. Dùng hàm mảng `COLLECT_LIST` kết hợp `ARRAY_JOIN`. |
| **Thay thế NULL khi nối chuỗi** | `ISNULL(' - ' + col, '')` | `COALESCE(CONCAT(' - ', col), '')` | `ISNULL` không tồn tại; `COALESCE` thay thế. Nối chuỗi bằng `+` với NULL cho ra NULL, nên dùng `CONCAT` để tự động bỏ qua NULL. |
| **Kiểu dữ liệu chuỗi** | `CAST(cnt AS NVARCHAR)` | `CAST(cnt AS STRING)` | Databricks không có `NVARCHAR`, dùng `STRING` thay thế. |
| **Toán tử nối chuỗi an toàn** | `'Extra ' + STRING_AGG(...)` | `CONCAT('Extra ', ARRAY_JOIN(...))` | `CONCAT` xử lý NULL tốt hơn, tránh mất cả chuỗi khi một thành phần NULL. |